# 👄 Lectura de labios multilingüe (VSR) — demo en Colab

**Qué hace:** recibe un vídeo de una persona hablando y **transcribe lo que dice SIN usar el audio**,
solo mirando el movimiento de los labios (*Visual Speech Recognition*). Con selector de idioma.

**Modelo:** [`mpc001/Visual_Speech_Recognition_for_Multiple_Languages`](https://github.com/mpc001/Visual_Speech_Recognition_for_Multiple_Languages)
— pesos preentrenados públicos. Solo **inferencia**, no se entrena nada.

---

### 📱 Cómo usarlo desde el móvil (Samsung), en orden
1. Abre este notebook en **Google Colab**.
2. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)** → Guardar.
3. Ejecuta las celdas **de arriba abajo, una a una**, esperando a que cada una termine (⏱️ el círculo se para).
4. Trabaja **por fases**: cada fase tiene una celda de **✅ VERIFICACIÓN**. No sigas si una verificación falla.
5. En la **Fase 4** aparecerá un **enlace público de Gradio** (`https://....gradio.live`). Ábrelo en el
   navegador del móvil y sube tu vídeo.

> ⚠️ **Honestidad primero.** Este notebook está construido con los archivos y enlaces **reales** del repo,
> pero **no ha sido ejecutado en Colab por quien lo escribió**. La lectura de labios es de las tareas más
> difíciles de IA: aun con buena luz y cara frontal, en español **falla casi la mitad de las palabras**.
> Es una **demo del estado del arte público**, no un transcriptor fiable. Al final tienes la tabla honesta
> de calidad por idioma y qué NO funciona.


## 🗺️ Fase 0 — Plan del notebook (léelo antes de ejecutar)

**Estructura (por fases, con verificación obligatoria en cada una):**
- **Fase 1** — Entorno + modelo español: comprobar GPU, instalar dependencias fijadas, clonar el repo,
  descargar los pesos de español. *Verifica:* versiones impresas + modelo español cargado en GPU sin error.
- **Fase 2** — Preprocesado del vídeo: detectar cara con **MediaPipe** (NO dlib), recortar la región de la
  boca (ROI). *Verifica:* se muestran 5 fotogramas del recorte de labios.
- **Fase 3** — Inferencia (español + inglés): transcribir un vídeo de cada uno. *Verifica:* texto obtenido
  vs texto real y % aproximado de acierto.
- **Fase 4** — Interfaz **Gradio** con `share=True`: subir vídeo, elegir idioma, ver recorte + transcripción.
- **Fase 5** — Resto de idiomas con pesos **bajo demanda**: francés, portugués, mandarín.

**Cómo se descargan los pesos:** el repo publica los pesos en **Google Drive** (enlaces `bit.ly` en su
tabla "model zoo"). En cada fase resolvemos el `bit.ly` → URL de Drive y descargamos con **`gdown`**, y
**solo el idioma activo** para no llenar el disco de Colab.

**Riesgos de compatibilidad conocidos (por eso vamos fase a fase):**
1. El repo se escribió para **Python 3.8 / PyTorch antiguo**; el Colab actual trae Python 3.12 / PyTorch 2.x.
   La instalación de MediaPipe/PyTorch es el punto más frágil.
2. La **estructura exacta al descomprimir** los `.zip` de Google Drive debe coincidir con las rutas que
   esperan los `.ini` (`benchmarks/<DATASET>/models/...`). Lo verificamos antes de cargar el modelo.
3. Google Drive a veces da **"límite de descarga superado"**; alternativa: Zenodo (doi 10.5281/zenodo.7065080).

> ❗ **El italiano NO tiene pesos públicos en este repo.** Su "model zoo" de VSR solo trae inglés, español,
> mandarín, francés y portugués. Lo dejaremos en el selector marcado como *no disponible* en vez de inventarlo.


## 🔧 Fase 1 — Entorno + modelo (español)

Objetivo: dejar el entorno listo y el **modelo español cargado en la GPU**.


In [ ]:
# 1.1 — ¿Tenemos GPU T4? (si esto falla: Entorno de ejecución → Cambiar tipo → GPU)
# No usamos torch aún porque todavía no está instalado; preguntamos directamente al sistema.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 1.2 — Instalar dependencias (versiones fijadas).
# NO tocamos el torch que ya trae Colab (reinstalarlo suele romper la GPU); usamos el suyo.
# Instalamos solo lo que el repo necesita y NO viene de serie.
# - mediapipe: detector de cara/labios SIN dlib (regla del proyecto).
# - hydra-core, av, scikit-image, opencv, scipy, six: dependencias del repo.
# - sentencepiece: tokenizador que usa el modelo de lenguaje.
# - gdown: para bajar los pesos de Google Drive.
%pip install -q \
  "mediapipe==0.10.9" \
  "hydra-core==1.3.2" \
  "av==11.0.0" \
  "scikit-image==0.22.0" \
  "opencv-python==4.9.0.80" \
  "scipy==1.11.4" \
  "six==1.16.0" \
  "sentencepiece==0.2.0" \
  "gdown==5.1.0"
print("✅ Dependencias instaladas. Si algo se queja de versiones, apúntalo y seguimos: probamos en la carga.")

In [ ]:
# 1.3 — Clonar el repositorio oficial del modelo.
import os
if not os.path.isdir("Visual_Speech_Recognition_for_Multiple_Languages"):
    !git clone --depth 1 https://github.com/mpc001/Visual_Speech_Recognition_for_Multiple_Languages
%cd Visual_Speech_Recognition_for_Multiple_Languages
!ls

In [ ]:
# 1.4 — Catálogo de idiomas: config real + enlaces reales de pesos (del "model zoo" del repo).
# Cada idioma necesita 2 descargas: el MODELO visual y el MODELO DE LENGUAJE.
# Con MediaPipe detectamos los labios al vuelo, así que NO hace falta el archivo de landmarks.
IDIOMAS = {
    "es": {"nombre": "Español",   "config": "configs/CMUMOSEAS_V_ES_WER44.5.ini",
           "modelo": "https://bit.ly/34MjWBW", "lm": "https://bit.ly/3rppyJN", "wer": 44.5},
    "en": {"nombre": "Inglés",    "config": "configs/LRS3_V_WER19.1.ini",
           "modelo": "https://bit.ly/3Bp4gjV", "lm": "https://bit.ly/3qzWKit", "wer": 19.1},
    "fr": {"nombre": "Francés",   "config": "configs/CMUMOSEAS_V_FR_WER58.6.ini",
           "modelo": "https://bit.ly/3Ik6owb", "lm": "https://bit.ly/3LDChSn", "wer": 58.6},
    "pt": {"nombre": "Portugués", "config": "configs/CMUMOSEAS_V_PT_WER51.4.ini",
           "modelo": "https://bit.ly/3HjXCgo", "lm": "https://bit.ly/3gPvneF", "wer": 51.4},
    "zh": {"nombre": "Mandarín",  "config": "configs/CMLR_V_WER8.0.ini",
           "modelo": "https://bit.ly/3fR8RkU", "lm": "https://bit.ly/3fPxXAJ", "wer": 8.0},
    # Italiano: sin pesos públicos en este repo -> no disponible (no lo inventamos).
    "it": {"nombre": "Italiano",  "config": None, "modelo": None, "lm": None, "wer": None},
}
print("Idiomas con pesos públicos:", [k for k,v in IDIOMAS.items() if v["config"]])
print("Sin pesos (no disponible):", [k for k,v in IDIOMAS.items() if not v["config"]])

In [ ]:
# 1.5 — Ayudante de descarga: resuelve el bit.ly -> Google Drive y baja con gdown.
# Los ficheros del repo son .zip que, al descomprimirse desde la RAÍZ del repo, crean la ruta
# que el .ini espera (benchmarks/<DATASET>/models/...). Verificamos que los ficheros aparezcan.
import os, zipfile, configparser, urllib.request, gdown

def _resolver_bitly(url):
    """bit.ly redirige a la URL real de Google Drive."""
    req = urllib.request.Request(url, method="HEAD")
    with urllib.request.urlopen(req) as r:
        return r.url

def _descargar_y_descomprimir(url, etiqueta):
    real = _resolver_bitly(url)
    print(f"  {etiqueta}: {url} -> {real}")
    destino = f"/tmp/{etiqueta}.zip"
    # --fuzzy deja que gdown saque el id del enlace de Drive aunque no sea el formato clásico.
    gdown.download(url=real, output=destino, quiet=False, fuzzy=True)
    if zipfile.is_zipfile(destino):
        with zipfile.ZipFile(destino) as z:
            z.extractall(".")   # descomprime respetando benchmarks/<DATASET>/...
        print(f"  ✔ {etiqueta} descomprimido")
    else:
        print(f"  ⚠ {etiqueta} NO es un zip: míralo a mano ({destino})")

def preparar_idioma(cod):
    """Descarga modelo + modelo de lenguaje de un idioma (solo si faltan)."""
    info = IDIOMAS[cod]
    if not info["config"]:
        raise ValueError(f"{info['nombre']} no tiene pesos públicos en este repo.")
    cfg = configparser.ConfigParser(); cfg.read(info["config"])
    faltan = not (os.path.exists(cfg["model"]["model_path"]) and os.path.exists(cfg["model"]["rnnlm"]))
    if not faltan:
        print(f"✅ {info['nombre']}: pesos ya presentes."); return info["config"]
    print(f"⬇️  Descargando pesos de {info['nombre']}...")
    _descargar_y_descomprimir(info["modelo"], f"{cod}_modelo")
    _descargar_y_descomprimir(info["lm"],     f"{cod}_lm")
    # Comprobación final de que los .ini encuentran sus ficheros:
    ok = os.path.exists(cfg["model"]["model_path"]) and os.path.exists(cfg["model"]["rnnlm"])
    print(("✅" if ok else "❌"), f"{info['nombre']}: model_path={cfg['model']['model_path']} existe={os.path.exists(cfg['model']['model_path'])}")
    if not ok:
        print("   Revisa dónde se descomprimieron los ficheros (a veces el zip trae otra carpeta raíz).")
    return info["config"]

print("Ayudantes listos.")

In [ ]:
# 1.6 — Descargar SOLO el español (empezamos simple, como pide el plan).
config_es = preparar_idioma("es")

In [ ]:
# ✅ VERIFICACIÓN Fase 1 — versiones + el modelo español carga en GPU sin errores.
import torch, sys
print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA disponible:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "SIN GPU")
import mediapipe as mp; print("MediaPipe:", mp.__version__)

from pipelines.pipeline import InferencePipeline
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
# detector="mediapipe" (sin dlib) y face_track=True para que calcule los labios al vuelo.
pipeline_es = InferencePipeline(config_es, detector="mediapipe", face_track=True, device=DEVICE)
print("\n✅ Modelo ESPAÑOL cargado en", DEVICE, "— Fase 1 superada.")

## 🎞️ Fase 2 — Preprocesado del vídeo (recorte de labios / ROI)

El modelo no mira el vídeo entero: mira **solo un recuadro de la boca** (unos 96×96 píxeles, en blanco y
negro). El flujo es: fotogramas → detectar cara con **MediaPipe** → localizar los labios → recortar y
normalizar. Vamos a **ver 5 de esos recortes** para comprobar con los ojos que la boca está bien centrada.

> Necesitas un vídeo de prueba **frontal y con buena luz**. Súbelo a Colab (panel izquierdo → 📁 → subir) y
> pon su ruta en `VIDEO_PRUEBA`. Ideal: 1–4 segundos, la cara ocupando buena parte del cuadro.


In [ ]:
# 2.1 — Ruta de tu vídeo de prueba en ESPAÑOL (cámbiala por la tuya).
VIDEO_PRUEBA_ES = "/content/mi_video_es.mp4"   # <-- súbelo y ajusta la ruta
import os
assert os.path.exists(VIDEO_PRUEBA_ES), "No encuentro el vídeo: súbelo al panel de archivos y corrige la ruta."
print("Vídeo encontrado:", VIDEO_PRUEBA_ES)

In [ ]:
# 2.2 — Obtener el recorte de labios que REALMENTE usa el modelo y guardar 5 fotogramas.
# Usamos el mismo cargador de datos del repo (así vemos exactamente lo que "ve" la red).
import numpy as np, cv2
from pipelines.data.data_module import AVSRDataLoader

cargador = AVSRDataLoader(modality="video", detector="mediapipe")
# load_data devuelve el tensor de fotogramas de la boca (T, 96, 96) en escala de grises.
try:
    roi = cargador.load_data(VIDEO_PRUEBA_ES)
    roi = roi.numpy() if hasattr(roi, "numpy") else np.asarray(roi)
except TypeError:
    # Por si esta versión pide también landmarks (None = que los calcule MediaPipe):
    roi = np.asarray(cargador.load_data(VIDEO_PRUEBA_ES, None))

print("Forma del recorte (fotogramas, alto, ancho):", roi.shape)
# Cogemos 5 fotogramas repartidos y los pegamos en una tira horizontal.
idx = np.linspace(0, len(roi) - 1, 5).astype(int)
tira = np.hstack([roi[i] for i in idx])
cv2.imwrite("/content/roi_labios.png", tira)
print("✅ Guardado /content/roi_labios.png")

In [ ]:
# ✅ VERIFICACIÓN Fase 2 — mira los 5 recortes: ¿se ve la BOCA centrada y nítida?
from PIL import Image
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 3))
plt.imshow(Image.open("/content/roi_labios.png"), cmap="gray")
plt.title("5 recortes de labios (lo que ve el modelo)"); plt.axis("off"); plt.show()
print("Si NO se ve la boca centrada -> el vídeo es de perfil / mala luz / cara pequeña. Prueba otro.")

## 🗣️ Fase 3 — Inferencia (español + inglés)

Ahora transcribimos de verdad. El modelo convierte los recortes de labios en texto usando su decodificador
(atención + CTC) y un **modelo de lenguaje** que ayuda a elegir palabras plausibles.

> 🧠 **¿Qué es un visema y por qué /p/, /b/ y /m/ se confunden?**
> Un **visema** es "la forma que ponen los labios para un sonido". El problema: /p/, /b/ y /m/ se ven
> **idénticos** (los labios se juntan y se abren igual). Lo que los diferencia está **donde la cámara no
> llega**: /b/ hace vibrar la garganta, /m/ saca aire por la nariz, /p/ ni una cosa ni otra. Es como intentar
> distinguir a tres gemelos que **siempre visten igual**: por fuera no puedes; necesitarías oírlos. Por eso
> leer labios acierta la *forma* pero se equivoca de *letra* muy a menudo.


In [ ]:
# 3.1 — Descargar inglés y cargar su modelo (el español ya está de la Fase 1).
config_en = preparar_idioma("en")
pipeline_en = InferencePipeline(config_en, detector="mediapipe", face_track=True, device=DEVICE)
print("✅ Modelo INGLÉS cargado.")

In [ ]:
# 3.2 — Ruta de tu vídeo de prueba en INGLÉS (frontal, buena luz).
VIDEO_PRUEBA_EN = "/content/mi_video_en.mp4"   # <-- ajusta la ruta
import os
assert os.path.exists(VIDEO_PRUEBA_EN), "Sube un vídeo en inglés y corrige la ruta."
print("OK:", VIDEO_PRUEBA_EN)

In [ ]:
# 3.3 — Medir acierto de forma sencilla: % de palabras acertadas respecto al texto real.
def porcentaje_acierto(real, obtenido):
    r = real.lower().split(); o = set(obtenido.lower().split())
    if not r: return 0.0
    return 100.0 * sum(1 for p in r if p in o) / len(r)

In [ ]:
# ✅ VERIFICACIÓN Fase 3 — transcribir 1 vídeo ES y 1 EN; comparar con el texto real.
# Escribe lo que DE VERDAD se dice en cada vídeo para poder medir:
TEXTO_REAL_ES = "escribe aqui lo que se dice en el video en espanol"
TEXTO_REAL_EN = "write here what is said in the english video"

hyp_es = pipeline_es(VIDEO_PRUEBA_ES)
hyp_en = pipeline_en(VIDEO_PRUEBA_EN)

print("── ESPAÑOL ──")
print("  Real    :", TEXTO_REAL_ES)
print("  Obtenido:", hyp_es)
print(f"  Acierto aprox: {porcentaje_acierto(TEXTO_REAL_ES, hyp_es):.0f}%  (recuerda: el repo reporta ~44.5% de ERROR)")
print("── INGLÉS ──")
print("  Real    :", TEXTO_REAL_EN)
print("  Obtenido:", hyp_en)
print(f"  Acierto aprox: {porcentaje_acierto(TEXTO_REAL_EN, hyp_en):.0f}%  (el repo reporta ~19% de ERROR)")

## 🖥️ Fase 4 — Interfaz Gradio (enlace público para el móvil)

Montamos una web sencilla: subes un vídeo, eliges idioma, y te devuelve **el recorte de labios** (para que
veas que detectó bien la boca) y **la transcripción**. Se lanza con `share=True` para darte un enlace
`gradio.live` que abres desde el móvil.


In [ ]:
# 4.1 — Instalar Gradio.
%pip install -q "gradio==4.44.0"
print("✅ Gradio instalado.")

In [ ]:
# 4.2 — App Gradio. Cargamos cada modelo la PRIMERA vez que se usa (y lo guardamos en caché).
import gradio as gr, numpy as np, cv2
from PIL import Image
from pipelines.pipeline import InferencePipeline
from pipelines.data.data_module import AVSRDataLoader

_pipes = {"es": pipeline_es, "en": pipeline_en}   # ya cargados
_cargador = AVSRDataLoader(modality="video", detector="mediapipe")

# En esta Fase 4 solo activamos los DOS idiomas ya verificados (simplicidad primero).
ACTIVOS = {"es": "Español", "en": "Inglés"}

def _get_pipe(cod):
    if cod not in _pipes:
        cfg = preparar_idioma(cod)
        _pipes[cod] = InferencePipeline(cfg, detector="mediapipe", face_track=True, device=DEVICE)
    return _pipes[cod]

def _preview_labios(ruta):
    try:
        roi = _cargador.load_data(ruta)
    except TypeError:
        roi = _cargador.load_data(ruta, None)
    roi = np.asarray(roi.numpy() if hasattr(roi, "numpy") else roi)
    idx = np.linspace(0, len(roi) - 1, 5).astype(int)
    return Image.fromarray(np.hstack([roi[i] for i in idx]))

def transcribir(video, idioma_nombre):
    if not video:
        return None, "Sube un vídeo primero."
    cod = [k for k, v in ACTIVOS.items() if v == idioma_nombre][0]
    try:
        preview = _preview_labios(video)
    except Exception as e:
        return None, f"No pude detectar la boca (¿cara de perfil / poca luz?): {e}"
    texto = _get_pipe(cod)(video)
    return preview, texto

with gr.Blocks(title="Lectura de labios (VSR)") as demo:
    gr.Markdown("# 👄 Lectura de labios multilingüe\nSube un vídeo **frontal y con buena luz**. "
                "Transcribe SIN audio. Es una demo: se equivoca mucho, sobre todo en español.")
    with gr.Row():
        with gr.Column():
            v = gr.Video(label="Vídeo (.mp4 / .avi)")
            idi = gr.Dropdown(choices=list(ACTIVOS.values()), value="Español", label="Idioma")
            btn = gr.Button("Transcribir", variant="primary")
        with gr.Column():
            img = gr.Image(label="Recorte de labios que ve el modelo")
            out = gr.Textbox(label="Transcripción", lines=4)
    btn.click(transcribir, [v, idi], [img, out])

demo.launch(share=True)   # <-- te da el enlace https://....gradio.live

**✅ VERIFICACIÓN Fase 4:** abre el enlace `gradio.live` en el móvil, sube un vídeo en español y otro
en inglés, y comprueba que se ve el recorte de labios y sale una transcripción. Si el recorte sale en negro
o vacío, el vídeo no es válido (perfil / poca luz).


## 🌍 Fase 5 — Resto de idiomas (bajo demanda)

Añadimos **francés, portugués y mandarín**. Los pesos se descargan **la primera vez** que eliges ese idioma
(no antes, para no llenar el disco). El **italiano** aparece en la lista pero **desactivado**: este repo no
publica pesos de italiano y no vamos a inventarlos.


In [ ]:
# 5.1 — Ampliar el selector. fr/pt/zh se descargan al usarlos por primera vez.
ACTIVOS = {"es": "Español", "en": "Inglés", "fr": "Francés", "pt": "Portugués", "zh": "Mandarín"}
# Italiano queda fuera a propósito: sin pesos públicos en este repo.
print("Idiomas del selector:", list(ACTIVOS.values()))
print("Nota: Italiano NO se incluye (no hay pesos públicos).")

In [ ]:
# 5.2 — Relanzar la app con todos los idiomas. La función transcribir() ya descarga bajo demanda.
demo.close()   # cerramos la instancia anterior
demo.launch(share=True)

**✅ VERIFICACIÓN Fase 5:** prueba **al menos un vídeo por idioma** (fr, pt, zh). La primera vez que
elijas uno nuevo tardará más (descarga sus pesos). Anota qué idioma probaste y el resultado.


## 📊 Calidad esperada por idioma (datos del propio repo) y qué NO funciona

**Tasa de ERROR de palabra (WER) que reporta el repo** — cuanto más bajo, mejor:

| Idioma    | Config usada                    | Error aprox. | Lectura honesta |
|-----------|---------------------------------|:-----------:|-----------------|
| Inglés    | `LRS3_V_WER19.1.ini`            | ~19–20%     | El mejor; aun así 1 de cada 5 palabras mal |
| Español   | `CMUMOSEAS_V_ES_WER44.5.ini`    | ~44.5%      | **Falla casi la mitad de las palabras** |
| Portugués | `CMUMOSEAS_V_PT_WER51.4.ini`    | ~51%        | Falla más de la mitad |
| Francés   | `CMUMOSEAS_V_FR_WER58.6.ini`    | ~59%        | Falla la mayoría |
| Mandarín  | `CMLR_V_WER8.0.ini`             | ~8% (por carácter) | Bajo por medirse por carácter, no por palabra |
| Italiano  | — no disponible —               | —           | **Sin pesos públicos en este repo** |

**Esto es una demo del estado del arte público, NO un transcriptor fiable.**

**Qué NO funciona (limitaciones reales):**
- **Caras de perfil** o giradas: necesita la boca de frente.
- **Mala luz** o boca en sombra: el recorte sale inservible.
- **Varios hablantes** en el cuadro: se lía; un solo rostro.
- Boca **tapada** (mano, micrófono, mascarilla), vídeo muy corto o muy comprimido.
- Sonidos que se ven igual (**/p/ /b/ /m/**, ver Fase 3): se confunden por diseño.

### Ideas si quieres mejor resultado
- Grabar **frontal, buena luz, vocalizando** y frases cortas.
- Para inglés los resultados son claramente mejores que para español.
- Un modelo **audiovisual** (con audio) acierta muchísimo más, pero eso ya no es "leer labios".
